# Heteroscedastic Multi-Output Bayes by Backprop on 2D Transport

    This notebook treats a 2D advection-diffusion transport field as a pointwise regression problem. Each sample predicts:

    - concentration $c(x, y)$
    - flux magnitude $|
abla c(x, y)|$

    together with input-dependent noise for both targets.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parents[1]
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import math
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from deepuq.methods import HeteroscedasticMultiOutputBayesByBackpropRegressor, predict_vi_uq, vi_elbo_step


## Transport field and output channels

    The notebook uses an analytic transport-like concentration field so the target generation stays inexpensive:

    $$
    c(x,y) = a e^{-\mathrm{Pe} x} \sin(\pi x) \sin(\pi y) + b x(1-x)y(1-y).
    $$

    The second output is the flux magnitude $|
abla c|$. Noise is larger near the inflow and in high-flux regions.


In [ ]:
torch.manual_seed(2)
grid = torch.linspace(0.0, 1.0, 36)
xx, yy = torch.meshgrid(grid, grid, indexing="ij")
coords = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)

def transport_outputs(xy, amp, bias, pe):
    x = xy[:, :1]
    y = xy[:, 1:2]
    concentration = amp * torch.exp(-pe * x) * torch.sin(math.pi * x) * torch.sin(math.pi * y) + bias * x * (1.0 - x) * y * (1.0 - y)
    dc_dx = amp * torch.exp(-pe * x) * (math.pi * torch.cos(math.pi * x) - pe * torch.sin(math.pi * x)) * torch.sin(math.pi * y) + bias * (1.0 - 2.0 * x) * y * (1.0 - y)
    dc_dy = amp * torch.exp(-pe * x) * torch.sin(math.pi * x) * math.pi * torch.cos(math.pi * y) + bias * x * (1.0 - x) * (1.0 - 2.0 * y)
    flux = torch.sqrt(dc_dx**2 + dc_dy**2 + 1e-8)
    return concentration, flux

def build_dataset(n_fields, ood=False):
    rows = []
    targets = []
    for _ in range(n_fields):
        if ood:
            amp = torch.empty(1).uniform_(1.2, 1.8).item()
            bias = torch.empty(1).uniform_(0.3, 0.7).item()
            pe = torch.empty(1).uniform_(2.0, 3.2).item()
        else:
            amp = torch.empty(1).uniform_(0.5, 1.1).item()
            bias = torch.empty(1).uniform_(0.0, 0.3).item()
            pe = torch.empty(1).uniform_(0.4, 1.6).item()
        concentration, flux = transport_outputs(coords, amp, bias, pe)
        sigma_c = 0.01 + 0.03 * torch.exp(-4.0 * coords[:, :1])
        sigma_f = 0.015 + 0.03 * (flux / flux.max())
        noisy_c = concentration + sigma_c * torch.randn_like(concentration)
        noisy_f = flux + sigma_f * torch.randn_like(flux)
        features = torch.cat(
            [
                coords,
                torch.full((coords.shape[0], 1), amp),
                torch.full((coords.shape[0], 1), bias),
                torch.full((coords.shape[0], 1), pe),
            ],
            dim=1,
        )
        rows.append(features)
        targets.append(torch.cat([noisy_c, noisy_f], dim=1))
    return torch.cat(rows, dim=0), torch.cat(targets, dim=0)

x_train, y_train = build_dataset(48)
x_test, y_test = build_dataset(12)
x_ood, y_ood = build_dataset(12, ood=True)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=256, shuffle=True)


In [ ]:
model = HeteroscedasticMultiOutputBayesByBackpropRegressor(input_dim=5, output_dim=2, hidden_dims=(96, 96), activation="tanh", prior_sigma=0.2)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
for _ in range(60):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = vi_elbo_step(model, xb, yb, num_batches=len(train_loader), kl_weight=0.01)
        loss.backward()
        optimizer.step()


In [ ]:
uq_test = predict_vi_uq(model, x_test, n_samples=30)
uq_ood = predict_vi_uq(model, x_ood, n_samples=30)

n_points = coords.shape[0]
n_test_fields = x_test.shape[0] // n_points
n_ood_fields = x_ood.shape[0] // n_points

test_std = uq_test.total_var.sqrt().reshape(n_test_fields, n_points, 2).mean(dim=0)
ood_std = uq_ood.total_var.sqrt().reshape(n_ood_fields, n_points, 2).mean(dim=0)

concentration_std = test_std[:, 0].reshape(xx.shape)
flux_std = test_std[:, 1].reshape(xx.shape)
concentration_std_ood = ood_std[:, 0].reshape(xx.shape)
flux_std_ood = ood_std[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
im0 = axes[0, 0].imshow(concentration_std, origin="lower", cmap="viridis")
axes[0, 0].set_title("Mean concentration predictive std (test)")
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)
im1 = axes[0, 1].imshow(flux_std, origin="lower", cmap="magma")
axes[0, 1].set_title("Mean flux predictive std (test)")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
im2 = axes[1, 0].imshow(concentration_std_ood, origin="lower", cmap="viridis")
axes[1, 0].set_title("Mean concentration predictive std (OOD)")
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)
im3 = axes[1, 1].imshow(flux_std_ood, origin="lower", cmap="magma")
axes[1, 1].set_title("Mean flux predictive std (OOD)")
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)
plt.tight_layout()

print(f"Test mean predictive std: {uq_test.total_var.sqrt().mean().item():.4f}")
print(f"OOD mean predictive std: {uq_ood.total_var.sqrt().mean().item():.4f}")
